# Wrangling Harmonic-EEG Data

This notebook gets the Harmonic-EEG dataset ready for analysis. The dataset has EEG recordings from 40 musicians, either classical musicians or jazz-trained improvisers, while they listened to three-chord progressions where the middle chord was occasionally changed. Before analyzing the EEG, we check who the participants are and that every participant's files are present and complete.

### Import libraries

Load the Python libraries used in this notebook: `pandas` for working with tables, `pathlib` for finding files in the `Data` folder, and `json` for reading the dataset's `.json` metadata files.

In [13]:
import pandas as pd
from pathlib import Path
import json

### Load and summarize participant information

`participants.tsv` has one row per participant with their ID, group (classical or improviser), handedness, age and gender. The cells below load it and summarize who is in the study:
- how many participants there are in total and in each group
- the average age and age range
- how many of each gender and handedness are in each group

In [10]:
# Load the TSV file into a DataFrame
participants_data = pd.read_csv('Data/participants.tsv', sep='\t')

# Convert to dataframe
df = pd.DataFrame(participants_data)

# Display the first few rows
df.head()

,participant_id,group,handedness,age,gender
0,sub-001,Cl,R,26,M
1,sub-002,Cl,R,26,M
2,sub-003,Cl,R,42,F
3,sub-004,Cl,R,24,F
4,sub-005,Imp,L,27,M


In [6]:
# Full names for the group codes used in the TSV
group_names = {"Cl": "classical musicians who don't improvise",
               "Imp": "improvisers (jazz-trained)"}

# Number of unique participants (participant_id is the primary key)
print(f"Participants: {df['participant_id'].nunique()}")

# Number of participants in each group
print("Group:")
for g, n in df["group"].value_counts().items():
    print(f"  {g} ({group_names[g]}): {n}")

# Mean and range of age
print(f"Age: mean {df['age'].mean():.1f}, range {df['age'].min()}–{df['age'].max()}")

# Number of participants of each gender
print("Gender:", df["gender"].value_counts().to_dict())

# Number of participants of each handedness
print("Handedness:", df["handedness"].value_counts().to_dict())

Participants: 40
Group:
  Imp (improvisers (jazz-trained)): 24
  Cl (classical musicians who don't improvise): 16
Age: mean 25.3, range 18–42
Gender: {'M': 24, 'F': 16}
Handedness: {'R': 31, 'A': 5, 'L': 4}


In [9]:
# Summary statistics for age
df["age"].agg(["count", "mean", "std", "min", "max"])

count    40.000000
mean     25.275000
std       5.514491
min      18.000000
max      42.000000
Name: age, dtype: float64

In [8]:
# Number of participants of each gender in each group
display(pd.crosstab(df["group"], df["gender"], margins=True))

# Number of participants of each handedness in each group
display(pd.crosstab(df["group"], df["handedness"], margins=True))

gender,F,M,All
group,,,
Cl,12,4,16
Imp,4,20,24
All,16,24,40


handedness,A,L,R,All
group,,,,
Cl,1,1,14,16
Imp,4,3,17,24
All,5,4,31,40


### Check that every participant has all their files

Each participant's `eeg` folder should contain the same set of files: the EEG recording itself (split into a `.set` header and an `.fdt` data file) plus smaller files describing the channels, electrodes and events. This cell lists every file with its size, then reports:
- any participant who is missing a file
- any EEG recording under 1 MB, which usually means it was never fully downloaded

In [11]:
root = Path("Data")

# List every file in each participant's eeg folder
rows = []
for file in sorted(root.glob("sub-*/eeg/*")):
    rows.append({
        # "sub-001_task-AuditoryOddballChords_events.tsv" -> "sub-001"
        "participant": file.name.split("_")[0],
        # "sub-001_task-AuditoryOddballChords_events.tsv" -> "events.tsv"
        "file": file.name.split("_")[-1],
        # Size in MB (0 if the file is a link to data that hasn't been downloaded)
        "size_MB": round(file.stat().st_size / 1e6, 2) if file.exists() else 0,
    })
files = pd.DataFrame(rows)

# Table with one row per participant and one column per file type (NaN = missing)
sizes = files.pivot(index="participant", columns="file", values="size_MB")

# Number of participants found
print(len(sizes), "participants found")

# Number of participants that have each file type (should all match the number above)
print("\nFiles per type:")
print(sizes.notna().sum())

# Participants missing at least one file type
missing = sizes[sizes.isna().any(axis=1)]
print("\nParticipants with missing files:")
print(missing if len(missing) else "  none")

# EEG recordings under 1 MB weren't fully downloaded
small = sizes.loc[(sizes["eeg.set"] < 1) | (sizes["eeg.fdt"] < 1), ["eeg.set", "eeg.fdt"]]
print("\nEEG recordings under 1 MB:")
print(small if len(small) else "  none")

40 participants found

Files per type:
file
channels.tsv        40
coordsystem.json    40
eeg.fdt             40
eeg.json            40
eeg.set             40
electrodes.tsv      40
events.json         40
events.tsv          40
dtype: int64

Participants with missing files:
  none

EEG recordings under 1 MB:
  none


### Check each participant's channels.tsv

`channels.tsv` lists the EEG channels (electrodes) in a recording. Under the BIDS standard it can have each channel's `name`, `type` (e.g. EEG or EOG), `units` (e.g. µV) and `status` (good or bad). This cell reads every participant's file and makes one row per participant showing:
- how many channels they have (should be 64)
- which columns their file has
- how many `type` and `units` values are missing (`n/a`)
- whether any channel name appears twice
- whether their channel list (names and order) matches everyone else's

In [15]:
# Read every participant's channels.tsv ("n/a" is read as missing)
channels = {f.name.split("_")[0]: pd.read_csv(f, sep="\t")
            for f in sorted(root.glob("sub-*/eeg/*_channels.tsv"))}

# The most common channel list (names in order), to compare each participant against
usual_names = pd.Series([tuple(c["name"]) for c in channels.values()]).mode()[0]

# One row per participant summarizing their channels.tsv
rows = []
for participant, c in channels.items():
    rows.append({
        "participant": participant,
        "n_channels": len(c),
        "columns": ", ".join(c.columns),
        "missing_type": c["type"].isna().sum(),
        "missing_units": c["units"].isna().sum(),
        "duplicate_names": c["name"].duplicated().sum(),
        "same_channels_as_others": tuple(c["name"]) == usual_names,
    })
channel_report = pd.DataFrame(rows).set_index("participant")

# How many participants share each combination of results (dropna=False keeps missing values)
channel_summary = channel_report.value_counts(dropna=False).to_frame("participants")
display(channel_summary)

# Full report only if participants' files differ
if len(channel_summary) > 1:
    display(channel_report)

,,,,,,participants
n_channels,columns,missing_type,missing_units,duplicate_names,same_channels_as_others,
64,"name, type, units",64,64,0,True,40


### Check each participant's coordsystem.json

`coordsystem.json` describes how to read the electrode positions in `electrodes.tsv`: which coordinate system they use and in what units. Under the BIDS standard it must have:
- `EEGCoordinateSystem`: the name of the coordinate system, or `Other` if it isn't a standard one
- `EEGCoordinateUnits`: the units of the positions (`m`, `mm`, `cm` or `n/a`)
- `EEGCoordinateSystemDescription`: a description of the system, required when it is `Other`

This cell reads every participant's file into one row per participant with one column per field (NaN = field missing). If every participant's file is the same, only a one-line summary is shown.

In [16]:
# Read every participant's coordsystem.json (one row per participant, one column per field)
rows = []
for f in sorted(root.glob("sub-*/eeg/*_coordsystem.json")):
    with open(f) as file:
        row = json.load(file)
    row["participant"] = f.name.split("_")[0]
    rows.append(row)
coord_report = pd.DataFrame(rows).set_index("participant")

# How many participants share each combination of values (dropna=False keeps missing fields)
coord_summary = coord_report.value_counts(dropna=False).to_frame("participants")
display(coord_summary)

# Full report only if participants' files differ
if len(coord_summary) > 1:
    display(coord_report)

,,,participants
EEGCoordinateUnits,EEGCoordinateSystem,EEGCoordinateSystemDescription,
mm,Other,EEGLAB,40


### Check each participant's electrodes.tsv

`electrodes.tsv` gives the position of each electrode on the head: its `name` and its `x`, `y`, `z` coordinates, in the coordinate system and units described in `coordsystem.json`. This cell reads every participant's file and makes one row per participant showing:
- how many electrodes they have (should be 64)
- which columns their file has
- how many values are missing (blank or `n/a`)
- whether their electrode names (in order) match everyone else's
- whether their electrode positions (x, y, z for each electrode, in order) match everyone else's

If every participant's results are the same, only a one-line summary is shown.

In [17]:
# Read every participant's electrodes.tsv ("n/a" and blanks are read as missing)
electrodes = {f.name.split("_")[0]: pd.read_csv(f, sep="\t")
              for f in sorted(root.glob("sub-*/eeg/*_electrodes.tsv"))}

# The most common electrode name list (in order), to compare each participant against
usual_electrodes = pd.Series([tuple(e["name"]) for e in electrodes.values()]).mode()[0]

# The most common set of positions (x, y, z for each electrode, in order), to compare against
usual_positions = pd.Series([tuple(e[["x", "y", "z"]].itertuples(index=False))
                             for e in electrodes.values()]).mode()[0]

# One row per participant summarizing their electrodes.tsv
rows = []
for participant, e in electrodes.items():
    rows.append({
        "participant": participant,
        "n_electrodes": len(e),
        "columns": ", ".join(e.columns),
        "missing_values": e.isna().sum().sum(),
        "same_names_as_others": tuple(e["name"]) == usual_electrodes,
        "same_positions_as_others": tuple(e[["x", "y", "z"]].itertuples(index=False)) == usual_positions,
    })
electrode_report = pd.DataFrame(rows).set_index("participant")

# How many participants share each combination of results (dropna=False keeps missing values)
electrode_summary = electrode_report.value_counts(dropna=False).to_frame("participants")
display(electrode_summary)

# Full report only if participants' files differ
if len(electrode_summary) > 1:
    display(electrode_report)

,,,,,participants
n_electrodes,columns,missing_values,same_names_as_others,same_positions_as_others,
64,"name, x, y, z",0,True,True,40


### Check each participant's events.json

`events.json` is meant to explain the columns of `events.tsv`: what each column means and what each event code stands for (for example, that `fourRoot_S` is a standard trial whose middle chord is IV in root position). Under the BIDS standard this file is optional, and when present it should be a JSON object (`{...}`).

In this dataset every participant's file is an empty list (`[]`). So, the meaning of the event codes has to come from the dataset README and the paper instead.

This cell reads every participant's file and makes one row per participant showing what kind of JSON it holds (BIDS expects `dict`) and how many entries it has. If every participant's results are the same, only a one-line summary is shown.

In [18]:
# Read every participant's events.json and record what it holds
rows = []
for f in sorted(root.glob("sub-*/eeg/*_events.json")):
    with open(f) as file:
        contents = json.load(file)
    rows.append({
        "participant": f.name.split("_")[0],
        # "dict" for a JSON object {...}, "list" for a JSON list [...]
        "json_type": type(contents).__name__,
        "n_entries": len(contents),
    })
events_json_report = pd.DataFrame(rows).set_index("participant")

# How many participants share each combination of results (dropna=False keeps missing values)
events_json_summary = events_json_report.value_counts(dropna=False).to_frame("participants")
display(events_json_summary)

# Full report only if participants' files differ
if len(events_json_summary) > 1:
    display(events_json_report)

,,participants
json_type,n_entries,
list,0,40


### Check each participant's events.tsv

`events.tsv` lists everything that happened during the recording, one row per event: when it happened (`onset`, in seconds from the start of the recording), how long it lasted (`duration`) and what it was (`value`). In this dataset all the information is in the `value` column:
- each trial is three chords 400 ms apart: `two` (the ii chord), a middle chord, then `one` (the I chord)
- the middle chord's code says which chord it was and what kind of trial it was, e.g. `fourRoot_S` = IV in root position on a standard trial, `five6_deviantE...` = an exemplar deviant, `four6_deviantF...` = a function deviant
- codes like `Correct - Function!` and the `correct`/`incorrect` endings appear to record the participant's responses (to confirm against the paper)
- `16128` marks the start of a block, and `boundary` marks where EEGLAB joined two blocks of recording together

This cell reads every participant's file and makes one row per participant showing the number of events, which columns are completely empty, missing values in the other columns, whether the events are in time order, the time of the last event, the number of different codes, and how many standard, exemplar deviant and function deviant trials there are. Checks where every participant has the same result are shown once; checks where they differ are shown for every participant. A last table lists every event code, how many participants have it, and the fewest and most times it appears for one participant.

In [19]:
# Read every participant's events.tsv ("n/a" and blanks are read as missing)
events = {f.name.split("_")[0]: pd.read_csv(f, sep="\t")
          for f in sorted(root.glob("sub-*/eeg/*_events.tsv"))}

# One row per participant summarizing their events.tsv
rows = []
for participant, e in events.items():
    rows.append({
        "participant": participant,
        "n_events": len(e),
        "columns": ", ".join(e.columns),
        # Columns with no values at all
        "empty_columns": ", ".join(e.columns[e.isna().all()]),
        # Number of missing values in each of the other columns, e.g. "missing_duration"
        **e.loc[:, e.notna().any()].isna().sum().add_prefix("missing_"),
        "onsets_in_order": e["onset"].is_monotonic_increasing,
        "last_onset_s": round(e["onset"].max()),
        "n_codes": e["value"].nunique(),
        # Middle-chord codes: "_S" = standard, "deviantE" = exemplar, "deviantF" = function
        "n_standard": e["value"].str.contains("_S").sum(),
        "n_exemplar_deviant": e["value"].str.contains("deviantE").sum(),
        "n_function_deviant": e["value"].str.contains("deviantF").sum(),
        "n_boundaries": (e["value"] == "boundary").sum(),
    })
events_report = pd.DataFrame(rows).set_index("participant")

# Checks where every participant has the same result, shown once
same_for_all = events_report.nunique(dropna=False) == 1
display(events_report.loc[:, same_for_all].iloc[0].to_frame("all participants"))

# Checks where participants differ, shown for every participant
display(events_report.loc[:, ~same_for_all])

# Every event code: how many participants have it, and the fewest / most times it appears
all_events = pd.concat(events, names=["participant", "row"]).reset_index()
code_counts = pd.crosstab(all_events["value"], all_events["participant"])
display(pd.DataFrame({"participants_with_code": (code_counts > 0).sum(axis=1),
                      "min_count": code_counts.min(axis=1),
                      "max_count": code_counts.max(axis=1)}))

,all participants
columns,"onset, duration, sample, trial_type, response_..."
empty_columns,"sample, response_time, stim_file"
missing_onset,0
missing_duration,7
missing_trial_type,0
missing_value,0
onsets_in_order,True
n_standard,1220
n_exemplar_deviant,109
n_function_deviant,111


,n_events,last_onset_s,n_codes
participant,,,
sub-001,4553,2350,23
sub-002,4583,2396,30
sub-003,4521,2343,30
sub-004,4566,2385,28
sub-005,4501,2445,28
sub-006,4553,2348,27
sub-007,4552,2337,26
sub-008,4732,2340,31
sub-009,4530,2347,29


,participants_with_code,min_count,max_count
value,,,
16128,39,0,8
16228,1,0,1
16368,2,0,1
Correct - Exemplar!,40,5,108
Correct - Function!,40,44,110
Incorrect - Standard!,39,0,270
boundary,40,7,7
five6_S,40,244,305
five6_Sincorrect,30,0,61
